In [ ]:
Gesamtanzahl Records (total und Jahr)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Dataset Documentation Values") \
    .getOrCreate()

BASE_PATH = "/taxi/raw"
YEARS = list(range(2015, 2022))

year_counts = []

for year in YEARS:
    df_year = spark.read.parquet(f"{BASE_PATH}/{year}/*.parquet")
    count = df_year.count()
    year_counts.append((year, count))
    print(f"{year}: {count}")

year_counts_df = spark.createDataFrame(year_counts, ["year", "record_count"])
year_counts_df.show()

total_records = sum(count for _, count in year_counts)
print(f"Total records: {total_records}")

26/05/18 12:58:42 WARN Utils: Your hostname, bdlc-021 resolves to a loopback address: 127.0.1.1; using 10.176.129.84 instead (on interface ens3)
26/05/18 12:58:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 12:58:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

2015: 146039231
2016: 131131805
2017: 113500327
2018: 102871387
2019: 84598444


26/05/18 12:58:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


2020: 24649092
2021: 30904308


+----+------------+
|year|record_count|
+----+------------+
|2015|   146039231|
|2016|   131131805|
|2017|   113500327|
|2018|   102871387|
|2019|    84598444|
|2020|    24649092|
|2021|    30904308|
+----+------------+

Total records: 633694594


In [2]:
import subprocess

for year in YEARS:
    command = f"hdfs dfs -find {BASE_PATH}/{year} -name '*.parquet' | wc -l"
    result = subprocess.check_output(command, shell=True).decode().strip()
    print(f"{year}: {result} parquet files")

2015: 12 parquet files
2016: 12 parquet files
2017: 12 parquet files
2018: 12 parquet files
2019: 12 parquet files
2020: 12 parquet files
2021: 12 parquet files


In [3]:
command = "hdfs dfs -du -h -s /taxi/raw"
result = subprocess.check_output(command, shell=True).decode()
print(result)

8.4 G  8.4 G  /taxi/raw



In [4]:
command = "hdfs dfs -du -h /taxi/raw"
result = subprocess.check_output(command, shell=True).decode()
print(result)

1.9 G    1.9 G    /taxi/raw/2015
1.7 G    1.7 G    /taxi/raw/2016
1.5 G    1.5 G    /taxi/raw/2017
1.4 G    1.4 G    /taxi/raw/2018
1.2 G    1.2 G    /taxi/raw/2019
357.2 M  357.2 M  /taxi/raw/2020
457.4 M  457.4 M  /taxi/raw/2021



In [5]:
df_2021 = spark.read.parquet("/taxi/raw/2021/*.parquet")

relevant_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount"
]

schema_info = []

for field in df_2021.schema.fields:
    if field.name in relevant_columns:
        schema_info.append((
            field.name,
            field.dataType.simpleString(),
            field.nullable
        ))

schema_info_df = spark.createDataFrame(
    schema_info,
    ["column_name", "data_type", "nullable"]
)

schema_info_df.show(truncate=False)

+---------------------+-------------+--------+
|column_name          |data_type    |nullable|
+---------------------+-------------+--------+
|tpep_pickup_datetime |timestamp_ntz|true    |
|tpep_dropoff_datetime|timestamp_ntz|true    |
|passenger_count      |double       |true    |
|trip_distance        |double       |true    |
|PULocationID         |bigint       |true    |
|DOLocationID         |bigint       |true    |
|payment_type         |bigint       |true    |
|fare_amount          |double       |true    |
|tip_amount           |double       |true    |
|total_amount         |double       |true    |
+---------------------+-------------+--------+



In [6]:
command = "hdfs fsck /taxi/raw -files -blocks -locations | head -80"
result = subprocess.check_output(command, shell=True).decode()
print(result)

Connecting to namenode via http://10.176.129.84:9870/fsck?ugi=cluster&files=1&blocks=1&locations=1&path=%2Ftaxi%2Fraw


FSCK started by cluster (auth:SIMPLE) from /10.176.129.84 for path /taxi/raw at Mon May 18 13:00:29 CEST 2026

/taxi/raw <dir>
/taxi/raw/2015 <dir>
/taxi/raw/2015/yellow_tripdata_2015-01.parquet 175325767 bytes, replicated: replication=1, 2 block(s):  OK
0. BP-1261728217-127.0.1.1-1777563211888:blk_1073741843_1019 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[10.176.129.84:9866,DS-87479a1e-f708-4769-a59a-26fd956be71d,DISK]]
1. BP-1261728217-127.0.1.1-1777563211888:blk_1073741844_1020 len=41108039 Live_repl=1  [DatanodeInfoWithStorage[10.176.129.84:9866,DS-87479a1e-f708-4769-a59a-26fd956be71d,DISK]]

/taxi/raw/2015/yellow_tripdata_2015-02.parquet 171639741 bytes, replicated: replication=1, 2 block(s):  OK
0. BP-1261728217-127.0.1.1-1777563211888:blk_1073741845_1021 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[10.176.129.84:9866,DS-87479a1e-f708-4769-a59a-26fd956be71d,DISK]]
1. BP-1261728217-127.0.1.1-1777563211888:blk_1073741846_1022 len=37422013 Live_repl=1  [DatanodeInfoW

In [7]:
spark.stop()